# 1.2.1 Multi Model Benchmark

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.generators import Math2DGenerator
from src.noise import AdaptiveGaussianNoise
from src.shadows import BandShadow
from src.splitter import GapSplitter
from src.orchestrator import ExperimentOrchestrator
from src.visualization import plot_experiment_results

from src.models import (
    BaseModel,
    PolynomialRegressionModel, 
    DecisionTreeModel, 
    RandomForestModel,
    SVRModel
)

### 1. Definizioak

In [ ]:
curves = {
    "Linear": lambda x: 2*x + 5,
    "Quadratic": lambda x: 0.5*(x-10)**2 + 5, # Centrada en 10
    "Sinusoidal": lambda x: 10 * np.sin(0.5 * x) + 15,
    "Circular": lambda x: np.sqrt(np.maximum(0, 8**2 - (x-10)**2)) + 5
}

shadows = {
    "Small (2u)": (9, 11),
    "Medium (6u)": (7, 13),
    "Large (10u)": (5, 15)
}

models_config = [
    (lambda: SVRModel(kernel='rbf', C=100), "SVR (RBF)"),
    (lambda: PolynomialRegressionModel(degree=2), "Poly Degree 2"),
    (lambda: PolynomialRegressionModel(degree=6), "Poly Degree 6"),
    (lambda: DecisionTreeModel(max_depth=None), "DecisionTree (Full)"),
    (lambda: RandomForestModel(n_estimators=50, random_state=42), "RandomForest")
]

### 2. Exekuzio Begizta

In [ ]:
results_list = []
artifacts_storage = {}

n_samples = 400
x_range = (0, 20)
noise = AdaptiveGaussianNoise(noise_ratio=0.1, seed=42)
splitter = GapSplitter(test_size=0.2, random_state=42)

print("Starting Benchmark...")

for curve_name, func in curves.items():
    gen = Math2DGenerator(func, seed=42)
    
    for shadow_name, (s_min, s_max) in shadows.items():
        shadow = BandShadow(axis='x', min_val=s_min, max_val=s_max)
        
        for model_factory, model_name in models_config:
            model = model_factory()
            
            orchestrator = ExperimentOrchestrator(gen, splitter, model)
            res = orchestrator.run(n_samples, x_range, noise, shadow)
            
            exp_id = f"{curve_name} | {shadow_name} | {model_name}"
            
            results_list.append({
                "ID": exp_id,
                "Curve": curve_name,
                "Shadow Size": shadow_name,
                "Model": model_name,
                "MSE Shadow": res.metrics.mse_test_shadow,
                "MSE Visible": res.metrics.mse_test_visible
            })
            
            artifacts_storage[exp_id] = res
            
print(f"Benchmark finished. Total experiments: {len(results_list)}")

### 3. Analisia eta Ranking-ak

In [ ]:
df_results = pd.DataFrame(results_list)

df_sorted = df_results.sort_values(by="MSE Shadow", ascending=True)
top_5_global = df_sorted.head(5)
bottom_5_global = df_sorted.tail(5)
top_5_small = df_sorted[df_sorted["Shadow Size"] == "Small (2u)"].head(5)
top_5_medium = df_sorted[df_sorted["Shadow Size"] == "Medium (6u)"].head(5)
top_5_large = df_sorted[df_sorted["Shadow Size"] == "Large (10u)"].head(5)

print("Analisia bukatuta.")

### 4. Bistaraketa

In [ ]:
def plot_selected_experiments(selection_df, title_prefix=""):
    print(f"\n{'='*20} {title_prefix} {'='*20}")
    display(selection_df[['ID', 'MSE Shadow', 'MSE Visible']])
    
    for idx, row in selection_df.iterrows():
        exp_id = row['ID']
        mse_val = row['MSE Shadow']
        
        if exp_id in artifacts_storage:
            res = artifacts_storage[exp_id]
            
            plot_experiment_results(
                res.split_data,
                res.predictions,
                title=f"{title_prefix} #{idx+1}\n{exp_id}\nMSE Shadow: {mse_val:.4f}"
            )
        else:
            print(f"Error: No artifacts found for {exp_id}")

#### 4.1 Top 5 Global

In [ ]:
plot_selected_experiments(top_5_global.reset_index(), "TOP 5 GLOBAL")

#### 4.2 Top 5 Txarrenak Global

In [ ]:
plot_selected_experiments(bottom_5_global.reset_index(), "WORST 5 GLOBAL")

#### 4.3 Top 5 - Itzal txikia (2u)

In [ ]:
plot_selected_experiments(top_5_small.reset_index(), "BEST SMALL SHADOW")

#### 4.4 Top 5 - Itzal ertaina (6u)

In [ ]:
plot_selected_experiments(top_5_medium.reset_index(), "BEST MEDIUM SHADOW")

#### 4.5 Top 5 - Itzal handia (10u)

In [ ]:
plot_selected_experiments(top_5_large.reset_index(), "BEST LARGE SHADOW")